# 03 `optimizer.step()` 和 `zero_grad()`：坡度便条如何变成参数变化

上一课只做到了：

```text
loss.backward()
    ↓
把坡度写进 w.grad
```

这一课继续回答两个问题：

1. 谁真正修改了参数？
2. 为什么每一轮都要清空旧梯度？

## 知识点标注

- 行业状态：基础标准。PyTorch、TensorFlow 等训练框架都采用类似的‘算梯度 → 更新参数 → 清理梯度’循环。
- 现实用途：真实大模型只是把一个 `w` 换成数百万到数千亿个参数。
- 学习深度：能区分三个动作，并能解释不清空梯度时发生了什么；暂时不要求阅读优化器内部源码。
- 本课类型：概念演示。只创建一个参数，不使用 GPU，不训练 MokioMind。

## 先固定三个角色

把训练过程想成三个人接力：

```text
backward()       optimizer.step()       zero_grad()
算坡度            按坡度走一步             擦掉旧便条
写入 w.grad       修改 w                  清空 w.grad
不改 w            不重新计算坡度          不改 w
```

最容易混淆的地方：

```text
w.grad = 便条上的坡度
w      = 真正的参数数值
```

`backward()` 只填写便条；`optimizer.step()` 才会修改参数。

In [2]:
import torch

w = torch.tensor([1.0], requires_grad=True)
target = 3.0
optimizer = torch.optim.SGD([w], lr=0.1)

loss = (w - target) ** 2
print('参数 w =', w.item())
print('目标 target =', target)
print('loss =', loss.item())
print('w.grad =', w.grad)

参数 w = 1.0
目标 target = 3.0
loss = 4.0
w.grad = None


### 先预测

现在还没有运行 `backward()`，所以 `w.grad` 应该是 `None`。

此时：

```text
w = 1.0       真参数
loss = 4.0    当前错误程度
w.grad = None 还没有坡度便条
```

In [2]:
loss.backward()

print('backward 之后')
print('w =', w.item(), '   <- 参数改了吗？')
print('w.grad =', w.grad.item(), ' <- 坡度便条')
print('loss =', loss.item())

backward 之后
w = 1.0    <- 参数改了吗？
w.grad = -4.0  <- 坡度便条
loss = 4.0


## `optimizer.step()`：真正走一步

现在有：

```text
w = 1.0
w.grad = -4.0
学习率 = 0.1
```

优化器执行的核心关系仍然是：

```text
新参数 = 旧参数 - 学习率 × 梯度
新 w = 1.0 - 0.1 × (-4.0) = 1.4
```

这里不需要自己写 `w.data -= ...`；优化器替我们完成这一步。

In [3]:
optimizer.step()
loss_after_step = (w - target) ** 2

print('step 之后 w =', w.item())
print('重新计算的 loss =', loss_after_step.item())
print('旧 loss 4.0 → 新 loss 2.56')

step 之后 w = 1.399999976158142
重新计算的 loss = 2.56000018119812
旧 loss 4.0 → 新 loss 2.56


## `zero_grad()`：为什么要擦掉旧便条

PyTorch 默认会把梯度累加。它不是每次 `backward()` 都覆盖旧值。

下面故意不调用 `zero_grad()`：

```text
第 1 次 backward：w.grad = -4.0
第 1 次 step：w 变成 1.4
第 2 次 loss 的新坡度：-3.2
如果不清空：-4.0 + (-3.2) = -7.2
```

第二次更新就会误把两轮的坡度一起拿来走，步子变大了。

In [6]:
w_bad = torch.tensor([1.0], requires_grad=True)
optimizer_bad = torch.optim.SGD([w_bad], lr=0.1)

loss1 = (w_bad - target) ** 2
loss1.backward()
print('第 1 次 backward，grad =', w_bad.grad.item())
optimizer_bad.step()

loss2 = (w_bad - target) ** 2
loss2.backward()  # 故意不 zero_grad
print('第 2 次 backward，但没清空，grad =', w_bad.grad.item())
optimizer_bad.step()
print('错误示范最终 w =', w_bad.item())

第 1 次 backward，grad = -4.0
第 2 次 backward，但没清空，grad = -7.199999809265137
错误示范最终 w = 2.119999885559082


## 正常循环：每轮使用自己的坡度

每一轮都按这个顺序：

```text
zero_grad()  →  forward 计算 loss  →  backward()  →  step()
清旧便条       产生当前 loss          写新便条       按新便条走一步
```

`zero_grad()` 放在这一轮开始或上一轮 `step()` 后都可以；关键是：本轮 `backward()` 前，不能残留上一轮的梯度。

In [7]:
w_good = torch.tensor([1.0], requires_grad=True)
optimizer_good = torch.optim.SGD([w_good], lr=0.1)

for round_id in range(1, 3):
    optimizer_good.zero_grad()
    loss = (w_good - target) ** 2
    loss.backward()
    print(f'第 {round_id} 轮：w={w_good.item():.2f}, loss={loss.item():.2f}, grad={w_good.grad.item():+.2f}')
    optimizer_good.step()

print(f'更新两轮后：w={w_good.item():.2f}, loss={(w_good - target).pow(2).item():.2f}')

第 1 轮：w=1.00, loss=4.00, grad=-4.00
第 2 轮：w=1.40, loss=2.56, grad=-3.20
更新两轮后：w=1.72, loss=1.64


## 回到 MokioMind 的训练代码

项目的完整代码还加入了混合精度和梯度累积，但角色没有变：

```python
res = model(input_ids, labels=labels, attention_mask=attention_mask)
loss = res.loss + res.aux_loss
scaler.scale(loss).backward()  # 计算并累积梯度
scaler.step(optimizer)         # 真正更新参数
scaler.update()
optimizer.zero_grad(set_to_none=True)  # 清空，准备下一轮
```

在项目中因为有 `accumulation_steps`，`step()` 和 `zero_grad()` 会按累积边界执行，不一定每个小批次都执行；这是训练速度和显存的工程安排，不改变三个动作各自的职责。

## 验收题

先自己回答，再运行或回看代码：

1. `backward()`、`optimizer.step()`、`zero_grad()` 分别修改什么？
2. 为什么第二次 `backward()` 前要清空第一次的 `w.grad`？
3. 如果 `w.grad=-4`、学习率是 `0.1`，`optimizer.step()` 后 `w` 为什么从 `1.0` 变成 `1.4`？
4. `optimizer.step()` 修改的是 `logits`，还是模型参数？